# 2026 Project

  
The dataset student_dropout_risk_2026.csv.gz contains one row for each
university student (identified by `person_id`), with demographic,
economic, academic and behavioural information.
The column `dropout` is `1` if the student has dropped out, `0` otherwise.
All columns whose values are `0`/`1` are boolean flags.

## Notes

- It is mandatory to use GitHub for developing the project.

- The project must be a Jupyter notebook.

- There is no restriction on the libraries that can be used, nor on the Python version.

- All questions on the project **must** be asked in the Discussion forum on the course website.

- 1,2, or 3 students can be in each group. You must create the groups by yourself.

- Avoid explicit `for` loops over the rows of the dataframe, unless the exercise explicitly asks for them.

## Student
Matteo Coppola
900166 , m.coppola42@campus.unimib.it

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('student_dropout_risk_2026.csv') #load the dataset 
print(df.shape)
df.head()

(100000, 34)


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,...,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,...,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,...,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,4,7.4,1,0,7,0,10,24,5.8,0


## 1.

Extract all students that study `Online`, have an `attendance_rate`
smaller than 50, and have more than 5 `missed_deadlines`.
Store them in a dataframe called `at_risk`.

In [3]:
at_risk = df[(df['study_mode']=='Online') & (df['attendance_rate'] < 50) & (df['missed_deadlines']>5)]
print(at_risk.shape)
at_risk.head(3)

(197, 34)


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
478,479,22,Male,Germany,0,1,High,0,0,1,...,0,13.3,1,8,8,1,3,30,6.1,1
581,582,17,Female,India,1,0,Low,1,1,1,...,3,12.4,1,11,6,0,8,0,7.1,1
642,643,29,Male,Mexico,0,0,Low,1,1,1,...,4,14.3,0,9,10,0,7,9,4.0,1


## 2.

Then compute, for each `course_type`, how many of those students there
are and what fraction of them has dropped out. Store the result in a
dataframe called `at_risk_by_course`, with columns `students` and
`dropout_rate`, sorted by decreasing `dropout_rate`.

In [4]:
at_risk_by_course = at_risk.groupby('course_type')[['person_id']].count()  # double brackets: the result stays a DataFrame instead of becoming a Series
at_risk_by_course.rename(columns={'person_id': 'students'}, inplace=True)  # this works only on a DataFrame
at_risk_by_course['dropout_rate'] = at_risk.groupby('course_type')['dropout'].mean()  # values are matched by index (course_type), the same as the groupby above
at_risk_by_course.sort_values('dropout_rate', ascending=False, inplace=True)
at_risk_by_course.reset_index(inplace=True)  # course_type goes back to being a column instead of the index
at_risk_by_course

,course_type,students,dropout_rate
0,Law & Social Sciences,29,0.896552
1,STEM,47,0.765957
2,Business,47,0.765957
3,Arts & Humanities,40,0.750000
4,Health Sciences,34,0.647059


## 3.

Add to the dataframe the following columns:

- `pass_rate`: the ratio between `units_passed` and `units_enrolled`;

- `gpa_change`: the difference between `gpa_semester_2` and `gpa_semester_1`;

- `gpa_trend`: a string that is `"improving"` if `gpa_change` is larger than 0.5, `"declining"` if `gpa_change` is smaller than -0.5, and `"stable"` otherwise.

In [5]:
df[['gpa_semester_1', 'gpa_semester_2']].isna().sum()  # check for missing GPA values before computing gpa_change

gpa_semester_1    0
gpa_semester_2    0
dtype: int64

In [6]:
(df['units_enrolled'] == 0).sum()  # check that no student has 0 units enrolled, to avoid division by zero in pass_rate

np.int64(0)

In [7]:
df['pass_rate'] = df['units_passed']/df['units_enrolled'] # I checked above that this computation is safe: every student is enrolled in at least one unit
df['gpa_change'] = (df['gpa_semester_2'] - df['gpa_semester_1']).round(2)  # I rounded this computation to avoid float errors (e.g. 3.6 - 3.1 = 0.5000000000000004); GPA values have at most 2 decimal places, so no information is lost 

# df['gpa_trend'] = 'stable'  # vectorized method: it is the fastest, but I find the function with apply more readable
# df.loc[df['gpa_change'] > 0.5, 'gpa_trend'] = 'improving'
# df.loc[df['gpa_change'] < -0.5, 'gpa_trend'] = 'declining'

#df['gpa_trend'] = ['improving' if x >0.5  else('declining' if x < -0.5 else 'stable')for x in df['gpa_change']] # alternative with list comprehension, similar speed to apply

def gpa_trend(x):
    if x > 0.5:
        return 'improving'
    elif x < -0.5:
        return 'declining'
    else:
        return 'stable'

df['gpa_trend'] = df['gpa_change'].apply(gpa_trend)  # I chose apply for readability, knowing it is slower than the vectorized .loc

df

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,3,8,0,6,37,5.7,0,1.000000,-0.89,declining
1,2,18,Other,USA,1,1,High,1,1,0,...,1,7,0,6,15,5.4,1,0.785714,-0.03,stable
2,3,24,Other,USA,0,0,High,0,1,1,...,0,2,0,3,4,4.3,0,0.823529,-0.70,declining
3,4,25,Female,USA,0,1,Middle,1,1,0,...,1,5,0,7,10,6.7,0,0.846154,-0.41,stable
4,5,15,Female,Canada,0,0,Low,1,1,0,...,0,7,0,10,24,5.8,0,0.875000,-1.05,declining
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,25,Female,USA,0,0,Middle,1,1,1,...,0,4,0,6,11,5.9,0,1.000000,-0.98,declining
99996,99997,24,Female,India,0,1,Middle,0,1,1,...,3,7,1,5,21,5.5,1,0.500000,0.21,stable
99997,99998,27,Female,USA,0,0,Middle,1,1,1,...,2,2,0,6,29,5.8,0,1.000000,-0.57,declining
99998,99999,21,Male,UK,0,0,High,0,1,0,...,2,4,1,10,43,6.2,0,0.714286,-0.99,declining


## 4.

Verify that, for every student, `units_passed + units_failed` equals
`units_enrolled`; if not, report the rows violating this condition.

In [8]:
wrong_units = df[df['units_passed'] + df['units_failed'] != df['units_enrolled']]  # rows violating the condition
print(wrong_units.shape[0], 'row(s) violate the condition')
wrong_units

0 row(s) violate the condition


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend


## 5.

Compute the dropout rate for each value of `gpa_trend`, and the
`person_id` of the 10 students with the largest GPA drop.

In [9]:
dropout_rate_gpa_trend = df.groupby('gpa_trend')['dropout'].mean().reset_index()  # dropout is 0/1, so its mean is the share of students who dropped out
dropout_rate_gpa_trend.rename(columns={'dropout': 'dropout_rate'}, inplace=True)  # the column now holds a rate, not the original 0/1 values
dropout_rate_gpa_trend

,gpa_trend,dropout_rate
0,declining,0.264795
1,improving,0.589337
2,stable,0.391453


In [10]:
worst_10_gpa_change = df.sort_values('gpa_change')[['person_id' , 'gpa_change']].head(10) #sort_values sorts in ascending order by default
worst_10_gpa_change # note: 4 students are tied at -2.23 (positions 9-12), so only 2 of them are included

,person_id,gpa_change
23225,23226,-2.49
96224,96225,-2.40
88489,88490,-2.37
51824,51825,-2.35
58935,58936,-2.33
46714,46715,-2.32
43035,43036,-2.30
72663,72664,-2.27
71899,71900,-2.23
36339,36340,-2.23


## 6.

Partition the students into the following groups, according to
`weekly_work_hours`: `0`, `1-10`, `11-20`, `21-30`, `more than 30`
(all intervals include both extremes).

In [11]:
print(df['weekly_work_hours'].dtype)                      # int64 -> only integers
print(df['weekly_work_hours'].isna().sum())               # 0 -> no missing values
print(df['weekly_work_hours'].min(), df['weekly_work_hours'].max())  # 0 40 -> range of values

int64
0
0 40


In [12]:
def time_slot(hour):
    if hour == 0:
        return '0'
    elif 1 <= hour <= 10:
        return '1-10'
    elif 11 <= hour <= 20:
        return '11-20'
    elif 21 <= hour <= 30:
        return '21-30'
    else:
        return 'more than 30'
# weekly_work_hours contains only integers from 0 to 40 (checked), so every value falls in exactly one group

df['work_group'] = df['weekly_work_hours'].apply(time_slot)  # the new column goes in df
students_by_work_group = df.groupby('work_group')['person_id'].count().reset_index()
students_by_work_group.rename(columns={'person_id': 'total_people'}, inplace=True)
students_by_work_group

,work_group,total_people
0,0,51808
1,1-10,6750
2,11-20,23923
3,21-30,15673
4,more than 30,1846


## 7.

Build a risk score by summing:

- 2 points if the tuition is not up to date;

- 1 point if `stress_level` is at least 8;

- 1 point if `sleep_hours` is smaller than 5;

- 1 point if the student has a part-time job with more than 20 weekly work hours;

- 2 points if `attendance_rate` is smaller than 60;

- 1 point if `missed_deadlines` is larger than 3.

For each value of the risk score, compute the number of students and the
dropout rate.

In [13]:
# #vectorized alternative that is the fastest way to do it, but not the most readable
# df['risk_score'] = (
#     2 * (df['tuition_up_to_date'] == 0) +
#     1 * (df['stress_level'] >= 8) +
#     1 * (df['sleep_hours'] < 5) +
#     1 * ((df['part_time_job'] == 1) & (df['weekly_work_hours'] > 20)) +
#     2 * (df['attendance_rate'] < 60) +
#     1 * (df['missed_deadlines'] > 3)
# )
# df

# risk_summary = df.groupby('risk_score')['person_id'].count().reset_index()
# risk_summary.rename(columns={'person_id': 'students'}, inplace=True)
# risk_summary['dropout_rate'] = df.groupby('risk_score')['dropout'].mean().values

# risk_summary

In [14]:
def risk_score(x):
    risk = 0
    if x['tuition_up_to_date'] ==0 :
        risk+=2
    if x['stress_level'] >=8 :
        risk+=1
    if x['sleep_hours'] < 5 :
        risk+=1
    if (x['part_time_job'] == 1) and (x['weekly_work_hours'] > 20)  : 
        risk+=1
    if x['attendance_rate'] < 60 :
        risk+=2
    if x['missed_deadlines']>3 :
        risk+=1
    return risk 

In [15]:
df['risk_score'] = df.apply(risk_score, axis=1)  # axis=1 passes one row (one student) at a time, because the function needs several columns of the same student
risk_summary = df.groupby('risk_score')[['person_id']].count()  # double brackets: the result stays a DataFrame (like above)
risk_summary.rename(columns={'person_id': 'students'}, inplace=True)
risk_summary['dropout_rate'] = df.groupby('risk_score')['dropout'].mean() # matched by index (risk_score)
risk_summary.reset_index(inplace=True)
risk_summary # here the difference is bigger: about 1.4 s with apply vs almost 0 s with the vectorized method

,risk_score,students,dropout_rate
0,0,26955,0.152847
1,1,24115,0.252084
2,2,21984,0.337200
3,3,14557,0.452703
4,4,7613,0.577433
5,5,3320,0.673494
6,6,1149,0.794604
7,7,267,0.827715
8,8,40,0.900000


## 8.

Read the GDP per capita dataset published by Our World in Data
directly from the URL

```
https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true
```

and store it in a dataframe called `gdp`. The column `ny_gdp_pcap_pp_kd`
contains the GDP per capita.

Clean the dataset, removing the rows that do not correspond to a single country
(e.g. `World`, `High-income countries`).
For each country keep only the row with the most recent `year`.

In [ ]:
#gdp = pd.read_csv('https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true') 
#it doesn't work for me, I couldn't figure out why
gdp = pd.read_csv('gdp-per-capita-worldbank.csv')

gdp = gdp[gdp['owid_region'].notna()] # aggregates (World, income groups, regions) have no owid_region, so this keeps only single countries
gdp = gdp.loc[gdp.groupby('entity')['year'].idxmax()] # idxmax returns the row index of the most recent year for each country
print(gdp.shape)
gdp

## 9.

Join the GPD data with the students data.
Compute the dropout rate for each continent.

In [ ]:
set(df['nationality']) - set(gdp['entity'])  # set difference: countries in 'df' that have no match in 'gdp'

In [ ]:
check = pd.merge(df , gdp , left_on = 'nationality' , right_on='entity' , how='right') 
# Right join: it keeps all the rows of 'gdp'. The rows where 'entity' is not null
# and 'nationality' is null are the countries that don't have any student in 'df'.
check

In [ ]:
set(gdp['entity']) - set(check[(check['nationality'].isna()) & (check['entity'].notna())]['entity'])  
# countries that already match between 'gdp' and 'df' (USA and UK are not included yet)

## Matching country names between the datasets

1. The cell below was my first approach to the problem: I wanted to generalize as much as possible, so that I would not have to manually look for the differences between the two datasets.

2. After writing this function, I asked an LLM whether there were other ways to do it, and it suggested using set operations (set difference), which give the same result in a simpler and faster way.

3. I also wanted to generalize the renaming of the mismatched values, but I could not find a way to do it with the tools covered in class. Set operations and merges only compare exact strings, so they can tell me *which* values don't match, but not *what* they correspond to (e.g. 'USA' → 'United States'). The AI suggested some approaches based on additional libraries, but since they were not covered during the lessons, I preferred to do the mapping by hand. With only a few mismatched countries, I think this is also the simplest solution.

4. I also considered using regular expressions, but that would have meant writing a specific regex for each mismatched country. Creating two regexes just for 'USA' and 'UK' did not make sense to me: I would still have had to check how those countries were written in the other dataset, which means looking at the data manually anyway. Moreover, those regexes would only work for these two specific cases and would not be a general method for any new mismatch. For these reasons, I chose to rename them manually.

In [ ]:
# # function to check how many and which countries have different keys in the two dataframes
# nazioni= list(gdp['entity'])
# paesi= list(df.groupby('nationality')['nationality'].size().reset_index(name= 'conteggio')['nationality'])

# def checker(x , y):
#     lista_diversi= []
#     lista_uguali=[]
#     for i in x:
#         count=0
       
#         for j in y:
           
#             if i != j:
#                 count+=1
#         if count == len(y) :
#             lista_diversi.append(i)
#         else:
#                 lista_uguali.append(i)
#     return lista_diversi , lista_uguali
            
# paesi_diversi = checker(paesi,nazioni)[0]
# paesi_diversi

In [ ]:
df['nationality']= df['nationality'].replace({'USA' : 'United States' , 'UK' : 'United Kingdom'})

print('Unmatched nationalities:', set(df['nationality']) - set(gdp['entity']))  # expected: set() -> every nationality has a match in 'gdp'

dataframe_final = pd.merge(df , gdp , left_on = 'nationality' , right_on='entity') # inner join: no student is lost, since every nationality now matches

dropout_by_continent = dataframe_final.groupby('owid_region')['dropout'].mean().reset_index() 
dropout_by_continent.rename(columns={'owid_region' : 'continent' , 'dropout' : 'dropout_rate'} , inplace=True)
dropout_by_continent


The dropout rate is almost the same across continents (between 31.7% and 32.3%), even though the number of students per continent varies a lot (from about 1,000 in Oceania to about 38,000 in North America). So the continent of origin does not seem to affect dropout.